# AMEX Enterprise Credit Risk Platform
## Notebook 25 — Repository Packaging: GitHub, Kaggle & LinkedIn Portfolio Generation
### Phase 1 · Problem Statement 2: Risk Tier Classification (Final Notebook)

CRISP-DM stage: **Deployment / Packaging**. Notebook 25 of 25 -- the final notebook of Problem 2. Builds Problem 2's own packages exactly as Notebook 18 does for Problem 1, into their own `Problem2_Risk_Tier_Classification/` subfolder -- then goes one step further and assembles a **single combined portfolio** ("**AMEX RiskIQ: Enterprise Credit Risk Platform**") that holds both Problem 1 and Problem 2 as clearly named subfolders under one appealing, recruiter-facing title, ready to push as one GitHub repo / Kaggle project / LinkedIn showcase.

**What this notebook builds:**

- Problem 2's own **GitHub / Kaggle / LinkedIn** packages (`data/`, `notebooks/`, `src/`, `models/`, `reports/`, `docs/`, `assets/`, `.gitignore`, `README.md`, `requirements.txt`, `LICENSE`), following the exact same convention as Notebook 18 -- unchanged from earlier builds of this notebook.
- A **Master Portfolio** at `Repository_Packaging/AMEX_RiskIQ_Enterprise_Credit_Risk_Platform/` that combines Problem 1's real package (built by Notebook 18, if it has been run) and Problem 2's real package above into one repository, one Kaggle project, and one LinkedIn showcase -- each problem in its own `Problem1_Default_Prediction/` / `Problem2_Risk_Tier_Classification/` subfolder, plus a platform-wide root `README.md`, `LICENSE`, and a combined LinkedIn `Platform_Project_Showcase.docx` built from real metrics pulled from both problems. If Notebook 18 hasn't been run yet, that is recorded honestly (never faked) and Problem 1's subfolder is added automatically the next time this notebook re-runs after Notebook 18 completes.

**Same hard size-safety rule as Notebook 18:** every file is checked against a live, configurable size cap before being copied; a file that is missing or over cap is recorded honestly in a manifest, never silently dropped and never faked.

**Also included:** a real, live **code quality check** across every Problem 2 notebook and standalone script, and a fresh Problem-2-scoped end-to-end pipeline flow diagram.

**Deliverables:** `GitHub_Repository_Package/`, `Kaggle_Project_Package/`, `LinkedIn_Project_Showcase/`, `code_quality_report.csv`, `packaging_manifest.csv`, `problem2_flow_diagram.png`, `Repository_Packaging_Report.docx` under `Repository_Packaging/Problem2_Risk_Tier_Classification/`; plus the combined `AMEX_RiskIQ_Enterprise_Credit_Risk_Platform/` master portfolio (both problems, one brand) under `Repository_Packaging/`.

**Run the single code cell below, once.** Idempotent — every package folder (including the master portfolio) is fully cleared and rebuilt from scratch on every re-run, so it always reflects the platform's current real state.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOK 01 (SELF-HEALING)
# =============================================================================
import os
import sys
import json
import time
import shutil
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebook 01 (Self-Healing)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first -- "
                             f"this notebook reads its pillar directory map.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (_resource_limits.get("warp_thread_count") or PROJECT_CONFIG.get("warp_thread_count")
                      or PROJECT_CONFIG["hardware"].get("logical_cores_detected"))

# --- Self-heal: same pattern as every Problem 2 notebook. ---
_REQUIRED_PILLARS = {
    "risk_tier_policy": "Problem2_Risk_Tier_Classification/01_Risk_Tier_Policy",
    "risk_tier_modeling": "Problem2_Risk_Tier_Classification/02_Risk_Tier_Modeling",
    "risk_tier_validation": "Problem2_Risk_Tier_Classification/03_Risk_Tier_Validation",
    "risk_tier_deployment": "Problem2_Risk_Tier_Classification/04_Risk_Tier_Deployment",
    "risk_tier_monitoring": "Problem2_Risk_Tier_Classification/05_Risk_Tier_Monitoring",
    "risk_tier_reporting": "Problem2_Risk_Tier_Classification/06_Risk_Tier_Reporting",
    "risk_tier_packaging": "Problem2_Risk_Tier_Classification/07_Risk_Tier_Packaging",
}
_config_healed = False
for _key, _rel_path in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _rel_path
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json -- added automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print("\u2705 project_config.json updated in place -- no need to re-run Notebook 01.")

# --- Scoped under the SAME repository_packaging pillar Notebook 18 uses, but
#     in its own subfolder -- Problem 1's real packages are never touched. ---
REPO_PKG_DIR = PILLAR_DIRS["repository_packaging"] / "Problem2_Risk_Tier_Classification"
REPO_PKG_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_SUMMARIES = {}
for _p in sorted(ARTIFACTS_DIR.glob("notebook_*_summary.json")):
    try:
        _num = int(_p.stem.split("_")[1])
    except (IndexError, ValueError):
        continue
    with open(_p, "r", encoding="utf-8") as f:
        NOTEBOOK_SUMMARIES[_num] = json.load(f)

PROBLEM2_NOTEBOOK_NUMBERS = list(range(19, 25))  # 19-24; this notebook (25) is itself
_p2_found = [n for n in PROBLEM2_NOTEBOOK_NUMBERS if n in NOTEBOOK_SUMMARIES]
print(f"Problem 2 notebook summaries found: {sorted(n for n in NOTEBOOK_SUMMARIES if n in PROBLEM2_NOTEBOOK_NUMBERS)}")
print(f"Problem 2 pipeline coverage (Notebooks 19-24): {len(_p2_found)} / {len(PROBLEM2_NOTEBOOK_NUMBERS)} found")
_p2_notebook_files = sorted(NOTEBOOKS_DIR.glob("19_*.ipynb")) + sorted(NOTEBOOKS_DIR.glob("2[0-5]_*.ipynb"))
print(f"Real Problem 2 notebook files on disk: {len(_p2_notebook_files)}")
print(f"Problem 2 packaging will be written under: {REPO_PKG_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) + "\n"
                       f"Fix: pip install {' '.join(missing)}")

import ast
import io

try:
    from pyflakes.api import check as _pyflakes_check
    from pyflakes.reporter import Reporter as _PyflakesReporter
    _HAS_PYFLAKES = True
except ImportError:
    _HAS_PYFLAKES = False

print(f"pyflakes available for static analysis: {_HAS_PYFLAKES}"
      + ("" if _HAS_PYFLAKES else "  (optional -- syntax-only checking will be used instead)"))


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)
print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: PACKAGING SIZE-SAFETY POLICY & SAFE-COPY HELPER
# =============================================================================
_section("SECTION 3: Packaging Size-Safety Policy & Safe-Copy Helper")

# --- Same ASSUMPTION policy values as Notebook 18, for consistency. ---
PACKAGING_POLICY = {
    "github_max_file_size_mb": 20.0,
    "kaggle_max_file_size_mb": 50.0,
    "linkedin_max_file_size_mb": 10.0,
    "copy_raw_competition_data": False,
}

PACKAGING_MANIFEST = []


def _safe_copy(package: str, src_path: Path, dest_path: Path, max_mb: float, note: str = "") -> bool:
    if not src_path.exists():
        PACKAGING_MANIFEST.append({"package": package, "source": str(src_path), "dest": str(dest_path),
                                    "included": False, "size_mb": None,
                                    "reason": "source file not found (upstream notebook has not run yet)"})
        return False
    _size_mb = round(src_path.stat().st_size / 1e6, 3)
    if _size_mb > max_mb:
        PACKAGING_MANIFEST.append({"package": package, "source": str(src_path), "dest": str(dest_path),
                                    "included": False, "size_mb": _size_mb,
                                    "reason": f"exceeds {package} packaging cap of {max_mb} MB"})
        return False
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_path, dest_path)
    PACKAGING_MANIFEST.append({"package": package, "source": str(src_path), "dest": str(dest_path),
                                "included": True, "size_mb": _size_mb, "reason": note or "copied"})
    return True


for _k, _v in PACKAGING_POLICY.items():
    print(f"  {_k:32s}: {_v}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE CODE QUALITY CHECK -- EVERY REAL PROBLEM 2 NOTEBOOK & SCRIPT
# =============================================================================
_section("SECTION 4: Live Code Quality Check -- Every Real Problem 2 Notebook & Script")


def _pyflakes_warning_count(source: str, filename: str):
    if not _HAS_PYFLAKES:
        return None
    _out, _err = io.StringIO(), io.StringIO()
    _reporter = _PyflakesReporter(_out, _err)
    try:
        return _pyflakes_check(source, filename, _reporter)
    except Exception:
        return None


def _check_source(label: str, source: str, kind: str) -> dict:
    _lines = source.count("\n") + 1
    try:
        _tree = ast.parse(source, filename=label)
        _syntax_valid = True
        _n_functions = sum(1 for n in ast.walk(_tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)))
        _note = ""
    except SyntaxError as exc:
        _syntax_valid = False
        _n_functions = 0
        _note = f"SyntaxError: {exc}"
    return {"file": label, "type": kind, "lines": _lines, "syntax_valid": _syntax_valid,
            "functions_defined": _n_functions,
            "pyflakes_warnings": _pyflakes_warning_count(source, label) if _syntax_valid else None,
            "notes": _note}


CODE_QUALITY_ROWS = []

for _nb_path in _p2_notebook_files:
    try:
        with open(_nb_path, "r", encoding="utf-8") as f:
            _nb_json = json.load(f)
        _code_cells = [c for c in _nb_json.get("cells", []) if c.get("cell_type") == "code"]
        _source = "\n\n".join("".join(c.get("source", [])) for c in _code_cells)
        CODE_QUALITY_ROWS.append(_check_source(_nb_path.name, _source, "notebook"))
    except (json.JSONDecodeError, OSError) as exc:
        CODE_QUALITY_ROWS.append({"file": _nb_path.name, "type": "notebook", "lines": None,
                                   "syntax_valid": False, "functions_defined": None,
                                   "pyflakes_warnings": None, "notes": f"could not read/parse notebook: {exc}"})

_STANDALONE_SCRIPTS = [
    PILLAR_DIRS["risk_tier_deployment"] / "api" / "risk_tier_service.py",
    PILLAR_DIRS["risk_tier_monitoring"] / "risk_tier_monitoring_job.py",
]
for _sp in _STANDALONE_SCRIPTS:
    if _sp.exists():
        with open(_sp, "r", encoding="utf-8") as f:
            CODE_QUALITY_ROWS.append(_check_source(_sp.name, f.read(), "script"))
    else:
        CODE_QUALITY_ROWS.append({"file": _sp.name, "type": "script", "lines": None, "syntax_valid": None,
                                   "functions_defined": None, "pyflakes_warnings": None,
                                   "notes": "not found -- upstream notebook has not run yet"})

code_quality_df = pd.DataFrame(CODE_QUALITY_ROWS)
_n_checked = len(code_quality_df)
_n_valid = int((code_quality_df["syntax_valid"] == True).sum())
code_quality_path = REPO_PKG_DIR / "code_quality_report.csv"
code_quality_df.to_csv(code_quality_path, index=False)
print(code_quality_df.to_string(index=False))
print(f"\nSyntax-valid: {_n_valid} / {_n_checked}")
print(f"\u2705 Saved -> {code_quality_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: SCAFFOLD THE THREE PACKAGE DIRECTORIES (FULL REBUILD EACH RUN)
# =============================================================================
_section("SECTION 5: Scaffold The Three Package Directories (Full Rebuild Each Run)")

GITHUB_PKG_DIR = REPO_PKG_DIR / "GitHub_Repository_Package"
KAGGLE_PKG_DIR = REPO_PKG_DIR / "Kaggle_Project_Package"
LINKEDIN_PKG_DIR = REPO_PKG_DIR / "LinkedIn_Project_Showcase"

for _pkg_dir in (GITHUB_PKG_DIR, KAGGLE_PKG_DIR, LINKEDIN_PKG_DIR):
    if _pkg_dir.exists():
        shutil.rmtree(_pkg_dir)
    _pkg_dir.mkdir(parents=True, exist_ok=True)

print(f"\u2705 Cleared & re-created: {GITHUB_PKG_DIR}")
print(f"\u2705 Cleared & re-created: {KAGGLE_PKG_DIR}")
print(f"\u2705 Cleared & re-created: {LINKEDIN_PKG_DIR}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: "PROBLEM 2 AT A GLANCE" -- REAL HEADLINE METRICS FOR ALL THREE PACKAGES
# =============================================================================
_section("SECTION 6: Problem 2 At A Glance -- Real Headline Metrics")


def _g(n, *path, default="Not yet available -- run Notebook "):
    d = NOTEBOOK_SUMMARIES.get(n)
    if d is None:
        return f"{default}{n:02d}"
    cur = d
    for p in path:
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        else:
            return f"{default}{n:02d}"
    return cur


_RISK_TIER_POLICY_PATH = PILLAR_DIRS["risk_tier_policy"] / "risk_tier_policy.json"
_RISK_TIER_POLICY = {}
if _RISK_TIER_POLICY_PATH.exists():
    with open(_RISK_TIER_POLICY_PATH, "r", encoding="utf-8") as f:
        _RISK_TIER_POLICY = json.load(f)

GLANCE = {
    "champion_model": _g(19, "champion_model"),
    "champion_holdout_auc": _RISK_TIER_POLICY.get("champion_holdout_auc", "Not yet available -- run Notebook 19"),
    "champion_holdout_amex_metric": _RISK_TIER_POLICY.get("champion_holdout_amex_metric", "Not yet available -- run Notebook 19"),
    "n_tiers": _g(19, "n_tiers"),
    "primary_method": _g(19, "primary_method"),
    "primary_method_full_kpi_pass": _g(20, "primary_method_full_kpi_pass"),
    "chi_square_p_value": _g(21, "chi_square_p_value"),
    "cramers_v": _g(21, "cramers_v"),
    "fair_lending_testing_status": _g(21, "fair_lending_testing_status"),
    "api_self_test_passed": _g(22, "api_self_test_passed"),
    "monitoring_n_alerts": _g(23, "n_alerts"),
    "method_agreement_kappa": _g(23, "method_agreement_kappa"),
}
for _k, _v in GLANCE.items():
    print(f"  {_k:35s}: {_v}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: GITHUB PACKAGE -- FOLDER TREE, REAL FILE COPIES, .gitignore & LICENSE
# =============================================================================
_section("SECTION 7: GitHub Package -- Folder Tree, Real File Copies, .gitignore & LICENSE")

GH_CAP = PACKAGING_POLICY["github_max_file_size_mb"]

# --- notebooks/ -- every real Problem 2 notebook currently on disk ---
for _nb_path in _p2_notebook_files:
    _safe_copy("github", _nb_path, GITHUB_PKG_DIR / "notebooks" / _nb_path.name, GH_CAP)

# --- src/ -- the real, standalone code Problem 2 generated ---
_safe_copy("github", PILLAR_DIRS["risk_tier_deployment"] / "api" / "risk_tier_service.py",
           GITHUB_PKG_DIR / "src" / "fastapi_service" / "risk_tier_service.py", GH_CAP)
_safe_copy("github", PILLAR_DIRS["risk_tier_deployment"] / "api" / "requirements-api.txt",
           GITHUB_PKG_DIR / "src" / "fastapi_service" / "requirements-api.txt", GH_CAP)
_safe_copy("github", PILLAR_DIRS["risk_tier_deployment"] / "docker" / "Dockerfile",
           GITHUB_PKG_DIR / "src" / "docker" / "Dockerfile", GH_CAP)
_safe_copy("github", PILLAR_DIRS["risk_tier_deployment"] / "docker" / "docker-compose.yml",
           GITHUB_PKG_DIR / "src" / "docker" / "docker-compose.yml", GH_CAP)
_safe_copy("github", PILLAR_DIRS["risk_tier_deployment"] / "docker" / ".dockerignore",
           GITHUB_PKG_DIR / "src" / "docker" / ".dockerignore", GH_CAP)
_safe_copy("github", PILLAR_DIRS["risk_tier_monitoring"] / "risk_tier_monitoring_job.py",
           GITHUB_PKG_DIR / "src" / "monitoring" / "risk_tier_monitoring_job.py", GH_CAP)

# --- reports/ -- every real Word report from Notebooks 19-24, size-gated ---
_REPORT_SOURCES = [
    (PILLAR_DIRS["risk_tier_policy"] / "Risk_Tier_Policy_Charter.docx", "risk_tier_policy"),
    (PILLAR_DIRS["risk_tier_modeling"] / "Risk_Tier_Model_Development_Report.docx", "risk_tier_modeling"),
    (PILLAR_DIRS["risk_tier_validation"] / "Risk_Tier_Validation_Report.docx", "risk_tier_validation"),
    (PILLAR_DIRS["risk_tier_deployment"] / "Risk_Tier_Deployment_Report.docx", "risk_tier_deployment"),
    (PILLAR_DIRS["risk_tier_monitoring"] / "Risk_Tier_Monitoring_Report.docx", "risk_tier_monitoring"),
    (PILLAR_DIRS["risk_tier_reporting"] / "Risk_Tier_Comprehensive_Report.docx", "risk_tier_reporting"),
]
for _src, _subdir in _REPORT_SOURCES:
    _safe_copy("github", _src, GITHUB_PKG_DIR / "reports" / _subdir / _src.name, GH_CAP)

# --- docs/ -- the real policy JSON, rollup JSON, and this run's code quality report ---
_safe_copy("github", _RISK_TIER_POLICY_PATH, GITHUB_PKG_DIR / "docs" / "risk_tier_policy.json", GH_CAP)
_safe_copy("github", PILLAR_DIRS["risk_tier_reporting"] / "risk_tier_rollup.json",
           GITHUB_PKG_DIR / "docs" / "risk_tier_rollup.json", GH_CAP)
_safe_copy("github", code_quality_path, GITHUB_PKG_DIR / "docs" / "code_quality_report.csv", GH_CAP)

# --- assets/ -- the key real diagrams and charts for embedding in README.md ---
_ASSET_SOURCES = [
    PILLAR_DIRS["risk_tier_modeling"] / "bad_rate_by_tier_both_methods_chart.png",
    PILLAR_DIRS["risk_tier_modeling"] / "calibration_by_tier_chart.png",
    PILLAR_DIRS["risk_tier_validation"] / "bad_rate_confidence_intervals_chart.png",
    PILLAR_DIRS["risk_tier_monitoring"] / "method_agreement_heatmap_chart.png",
    PILLAR_DIRS["risk_tier_reporting"] / "final_bad_rate_by_tier_chart.png",
    PILLAR_DIRS["risk_tier_reporting"] / "notebook_completion_tracker_chart.png",
]
for _src in _ASSET_SOURCES:
    _safe_copy("github", _src, GITHUB_PKG_DIR / "assets" / _src.name, GH_CAP)

# --- models/ -- Problem 2 reuses Problem 1's real champion model; documented, not duplicated ---
_models_readme_lines = [
    "# Models", "",
    "Problem 2 (Risk Tier Classification) does **not** train a new model -- it reuses Problem 1's real, "
    f"saved champion model (**{GLANCE['champion_model']}**, holdout AUC {GLANCE['champion_holdout_auc']}, "
    f"holdout AMEX metric {GLANCE['champion_holdout_amex_metric']}) as-is, and adds a policy-based tier "
    "classification layer on top of its real predicted PD scores. See `docs/risk_tier_policy.json` for the "
    "real, saved bucketing policy (business-rule thresholds and quantile cutpoints) this layer applies -- "
    "that policy file, not a retrained model binary, is Problem 2's own core artifact.",
]
(GITHUB_PKG_DIR / "models").mkdir(parents=True, exist_ok=True)
with open(GITHUB_PKG_DIR / "models" / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_models_readme_lines) + "\n")

# --- data/ -- the real risk_tier_assignments.csv (a computed output, not raw competition data) ---
_safe_copy("github", PILLAR_DIRS["risk_tier_modeling"] / "risk_tier_assignments.csv",
           GITHUB_PKG_DIR / "data" / "risk_tier_assignments.csv", GH_CAP,
           note="real, computed output -- not raw competition data")
_data_readme = [
    "# Data", "",
    "`risk_tier_assignments.csv` (if present above) is this platform's own real, computed output -- per-customer "
    "predicted PD and tier assignment on the real, held-out Kaggle test split. The raw Kaggle American Express "
    "Default Prediction competition CSVs themselves are not redistributed here -- see Problem 1's "
    "`GitHub_Repository_Package/data/README.md` (built by Notebook 18) for how to obtain them.",
]
with open(GITHUB_PKG_DIR / "data" / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_data_readme) + "\n")

_gitignore_lines = [
    "# Python", "__pycache__/", "*.py[cod]", "*.egg-info/", ".Python", "build/", "dist/", "",
    "# Environments", ".env", ".venv", "venv/", "env/", "",
    "# Jupyter", ".ipynb_checkpoints/", "",
    "# OS", ".DS_Store", "Thumbs.db", "",
]
with open(GITHUB_PKG_DIR / ".gitignore", "w", encoding="utf-8") as f:
    f.write("\n".join(_gitignore_lines) + "\n")

_license_text = f"""MIT License

Copyright (c) {datetime.now().year} [Your Name]

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""
with open(GITHUB_PKG_DIR / "LICENSE", "w", encoding="utf-8") as f:
    f.write(_license_text)

_fallback_requirements = [
    "numpy", "pandas", "scikit-learn", "matplotlib", "psutil", "joblib", "python-docx", "scipy",
    "fastapi", "uvicorn", "pydantic", "pyyaml", "httpx",
]
with open(GITHUB_PKG_DIR / "requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(_fallback_requirements) + "\n")

print(f"\u2705 GitHub package populated under {GITHUB_PKG_DIR}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: GITHUB PACKAGE -- COMPREHENSIVE README.md (BUILT FROM REAL DATA)
# =============================================================================
_section("SECTION 8: GitHub Package -- Comprehensive README.md")


def _fmt_val(v, kind="num"):
    if isinstance(v, str):
        return v
    if kind == "pct" and isinstance(v, (int, float)):
        return f"{v:.2%}"
    if kind == "int" and isinstance(v, (int, float)):
        return f"{v:,.0f}"
    return str(v)


def _render_tree(root: Path, max_entries: int = 14, _prefix: str = "") -> list:
    lines = []
    try:
        entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    except FileNotFoundError:
        return lines
    shown = entries[:max_entries]
    for i, entry in enumerate(shown):
        connector = "\u2514\u2500\u2500 " if i == len(shown) - 1 and len(entries) <= max_entries else "\u251c\u2500\u2500 "
        label = entry.name + ("/" if entry.is_dir() else "")
        lines.append(f"{_prefix}{connector}{label}")
        if entry.is_dir():
            extension = "    " if connector.startswith("\u2514") else "\u2502   "
            lines.extend(_render_tree(entry, max_entries, _prefix + extension))
    if len(entries) > max_entries:
        lines.append(f"{_prefix}\u2514\u2500\u2500 ... (+{len(entries) - max_entries} more)")
    return lines


_repo_tree_lines = [f"{GITHUB_PKG_DIR.name}/"] + _render_tree(GITHUB_PKG_DIR)

_readme_lines = [
    "# AMEX Enterprise Credit Risk Platform -- Risk Tier Classification",
    "",
    "[![Python 3.11](https://img.shields.io/badge/python-3.11-blue.svg)]() "
    "[![CRISP-DM](https://img.shields.io/badge/methodology-CRISP--DM-informational.svg)]() "
    "[![License: MIT](https://img.shields.io/badge/license-MIT-green.svg)](LICENSE)",
    "",
    "Phase 1, Problem 2 of a 14-problem enterprise credit risk platform: a **7-notebook** build "
    "(Notebooks 19-25) that translates Problem 1's real, calibrated probability-of-default score "
    "into discrete, validated, deployed, and monitored risk tiers -- built end-to-end on the real "
    "Kaggle **American Express Default Prediction** dataset. Depends on Problem 1's real champion "
    "model (see the sibling GitHub package built by Problem 1's Notebook 18).",
    "",
    "## 1. Overview",
    "",
    f"- **Champion model reused (measured, Problem 1):** {_fmt_val(GLANCE['champion_model'])}",
    f"- **Champion holdout AUC (measured):** {_fmt_val(GLANCE['champion_holdout_auc'])}",
    f"- **Tiers defined:** {_fmt_val(GLANCE['n_tiers'])}, primary method: {_fmt_val(GLANCE['primary_method'])}",
    f"- **Primary method full KPI compliance:** {_fmt_val(GLANCE['primary_method_full_kpi_pass'])}",
    "",
    "## 2. Problem Statement",
    "",
    "A continuous PD score is not, by itself, an actionable underwriting or pricing instrument -- credit "
    "policy is written in discrete risk grades (e.g. 'Prime', 'Subprime'), each carrying its own approval "
    "rule, pricing, and credit-limit policy. This build defines a real, validated, monotonic mapping from "
    "the continuous PD score to a small set of risk tiers, and exposes that mapping as a live service.",
    "",
    "## 3. Approach & Methodology",
    "",
    "Notebook 19 (Business Understanding & Policy) -> Notebook 20 (Model Development -- two real tier-"
    "bucketing methods computed and compared) -> Notebook 21 (Independent Validation -- statistical "
    "backtesting, bootstrap CIs, an honest fair-lending data-limitation statement) -> Notebook 22 "
    "(Deployment -- a real FastAPI risk-tier service, live self-tested) -> Notebook 23 (Monitoring -- "
    "simulated monitoring windows, tier population stability) -> Notebook 24 (Comprehensive Reporting) "
    "-> Notebook 25 (this packaging notebook).",
    "",
    "## 4. Key Results (Real, Measured)",
    "",
    f"- Chi-square (tier vs. actual default), p-value: {_fmt_val(GLANCE['chi_square_p_value'])}",
    f"- Cram\u00e9r's V (effect size): {_fmt_val(GLANCE['cramers_v'])}",
    f"- Fair-lending / disparate-impact testing: {_fmt_val(GLANCE['fair_lending_testing_status'])}",
    f"- Live API self-test (Notebook 22): {_fmt_val(GLANCE['api_self_test_passed'])}",
    f"- Monitoring alerts (Notebook 23, this run): {_fmt_val(GLANCE['monitoring_n_alerts'])}",
    f"- Quantile-vs-business-rule method agreement (Cohen's kappa): {_fmt_val(GLANCE['method_agreement_kappa'])}",
    "",
    "![Bad rate by tier](assets/bad_rate_by_tier_both_methods_chart.png)",
    "![Confidence intervals](assets/bad_rate_confidence_intervals_chart.png)",
    "",
    "## 5. How to Run",
    "",
    "1. Complete Problem 1 (Notebooks 01-18) first -- Problem 2 has a hard dependency on Notebook 05's "
    "real champion model.",
    "2. Run `notebooks/19_risk_tier_business_understanding.ipynb` through "
    "`notebooks/25_risk_tier_packaging.ipynb` in numeric order -- each is a single, idempotent code cell.",
    "",
    "## 6. Project Structure",
    "",
    "This tree is real -- generated by walking this exact package folder, not hand-typed:",
    "",
    "```",
] + _repo_tree_lines + [
    "```",
    "",
    "## 7. Code Quality",
    "",
    f"{_n_valid} / {_n_checked} Problem 2 files passed a syntax check as of this run -- see "
    f"`docs/code_quality_report.csv`.",
    "",
    "## License",
    "",
    "MIT -- see `LICENSE`.",
    "",
    f"_Generated by 25_risk_tier_packaging.ipynb, {datetime.now().strftime('%Y-%m-%d %H:%M')}._",
    "",
]
GITHUB_README_CONTENT = "\n".join(_readme_lines)
github_readme_path = GITHUB_PKG_DIR / "README.md"
with open(github_readme_path, "w", encoding="utf-8") as f:
    f.write(GITHUB_README_CONTENT)
print(f"\u2705 Saved -> {github_readme_path}  ({len(GITHUB_README_CONTENT):,} characters)")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: KAGGLE PACKAGE -- CURATED NOTEBOOK MAPPING, SCRIPTS, MODELS & IMAGES
# =============================================================================
_section("SECTION 9: Kaggle Package -- Curated Notebook Mapping, Scripts, Models & Images")

KG_CAP = PACKAGING_POLICY["kaggle_max_file_size_mb"]

# --- Problem 2 is already a small, focused 6-notebook build (19-24) -- each
#     gets a clear Kaggle-style rename rather than being forced into an
#     artificially smaller count. ---
KAGGLE_NOTEBOOK_MAPPING = [
    {"kaggle_name": "01_policy_and_business_understanding.ipynb", "source_name": "19_risk_tier_business_understanding.ipynb", "kaggle_label": "Business Understanding & Tier Policy"},
    {"kaggle_name": "02_model_development.ipynb", "source_name": "20_risk_tier_model_development.ipynb", "kaggle_label": "Tier Model Development"},
    {"kaggle_name": "03_validation.ipynb", "source_name": "21_risk_tier_validation.ipynb", "kaggle_label": "Independent Validation"},
    {"kaggle_name": "04_deployment.ipynb", "source_name": "22_risk_tier_deployment.ipynb", "kaggle_label": "Deployment"},
    {"kaggle_name": "05_monitoring.ipynb", "source_name": "23_risk_tier_monitoring.ipynb", "kaggle_label": "Ongoing Monitoring"},
    {"kaggle_name": "06_results_and_reporting.ipynb", "source_name": "24_risk_tier_reporting.ipynb", "kaggle_label": "Results & Reporting"},
]
for _m in KAGGLE_NOTEBOOK_MAPPING:
    _src = NOTEBOOKS_DIR / _m["source_name"]
    _ok = _safe_copy("kaggle", _src, KAGGLE_PKG_DIR / "notebooks" / _m["kaggle_name"], KG_CAP,
                      note=f"renamed from real {_m['source_name']}")
    _m["included"] = _ok

_mapping_lines = ["# Notebook Mapping", "",
                   "Each file below is a **real, unmodified copy** of one of Problem 2's actual 6 substantive "
                   "notebooks (19-24), renamed for Kaggle's convention -- nothing here is a rewritten or "
                   "fabricated summary.", "",
                   "| Kaggle file | Real source notebook | Purpose |", "|---|---|---|"]
for _m in KAGGLE_NOTEBOOK_MAPPING:
    _status = "" if _m["included"] else "  _(not yet run -- excluded this build)_"
    _mapping_lines.append(f"| `{_m['kaggle_name']}` | `{_m['source_name']}` | {_m['kaggle_label']}{_status} |")
(KAGGLE_PKG_DIR / "notebooks").mkdir(parents=True, exist_ok=True)
with open(KAGGLE_PKG_DIR / "notebooks" / "NOTEBOOK_MAPPING.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_mapping_lines) + "\n")

_safe_copy("kaggle", PILLAR_DIRS["risk_tier_deployment"] / "api" / "risk_tier_service.py",
           KAGGLE_PKG_DIR / "scripts" / "risk_tier_service.py", KG_CAP,
           note="real FastAPI risk-tier serving code from Notebook 22")
_safe_copy("kaggle", PILLAR_DIRS["risk_tier_monitoring"] / "risk_tier_monitoring_job.py",
           KAGGLE_PKG_DIR / "scripts" / "risk_tier_monitoring_job.py", KG_CAP,
           note="real scheduled monitoring script from Notebook 23")

(KAGGLE_PKG_DIR / "models").mkdir(parents=True, exist_ok=True)
with open(KAGGLE_PKG_DIR / "models" / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_models_readme_lines) + "\n")
_safe_copy("kaggle", _RISK_TIER_POLICY_PATH, KAGGLE_PKG_DIR / "models" / "risk_tier_policy.json", KG_CAP)

_RESULT_CHARTS = [
    (PILLAR_DIRS["risk_tier_modeling"], "bad_rate_by_tier_both_methods_chart.png"),
    (PILLAR_DIRS["risk_tier_validation"], "bad_rate_confidence_intervals_chart.png"),
    (PILLAR_DIRS["risk_tier_monitoring"], "method_agreement_heatmap_chart.png"),
    (PILLAR_DIRS["risk_tier_reporting"], "final_bad_rate_by_tier_chart.png"),
]
for _dir, _c in _RESULT_CHARTS:
    _safe_copy("kaggle", _dir / _c, KAGGLE_PKG_DIR / "images" / "results" / _c, KG_CAP)

(KAGGLE_PKG_DIR / "data" / "input").mkdir(parents=True, exist_ok=True)
with open(KAGGLE_PKG_DIR / "data" / "input" / "README.md", "w", encoding="utf-8") as f:
    f.write("Original Kaggle competition data -- not redistributed here. See Problem 1's package.\n")
_safe_copy("kaggle", PILLAR_DIRS["risk_tier_modeling"] / "risk_tier_assignments.csv",
           KAGGLE_PKG_DIR / "data" / "output" / "risk_tier_assignments.csv", KG_CAP)

print(f"\u2705 Kaggle package populated under {KAGGLE_PKG_DIR}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: KAGGLE PACKAGE -- README.md, requirements.txt & environment.yml
# =============================================================================
_section("SECTION 10: Kaggle Package -- README.md, requirements.txt & environment.yml")

_kaggle_readme_lines = [
    "# AMEX Risk Tier Classification (Kaggle Project Package)",
    "",
    "## Overview",
    "",
    f"Reuses Problem 1's real champion model (**{_fmt_val(GLANCE['champion_model'])}**) to classify "
    f"customers into **{_fmt_val(GLANCE['n_tiers'])}** real, validated risk tiers.",
    "", "## Dataset", "",
    "Kaggle competition: American Express Default Prediction "
    "(https://www.kaggle.com/competitions/amex-default-prediction). Raw data is not redistributed here.",
    "", "## Methodology", "",
    "See `notebooks/NOTEBOOK_MAPPING.md` for exactly which real, full platform notebook each file "
    "here is drawn from.",
    "", "## Results", "",
    f"- Chi-square p-value (tier vs. actual default): {_fmt_val(GLANCE['chi_square_p_value'])}",
    f"- Fair-lending testing status: {_fmt_val(GLANCE['fair_lending_testing_status'])}",
    "", "## How to Run", "",
    "1. Install dependencies from `requirements.txt` (or `environment.yml` for conda).",
    "2. Complete Problem 1 first, then run the notebooks in `notebooks/` in the order shown in "
    "`NOTEBOOK_MAPPING.md`.",
    "", "## Acknowledgements", "",
    "Built on the Kaggle American Express Default Prediction competition dataset.",
    "", f"_Generated by 25_risk_tier_packaging.ipynb, {datetime.now().strftime('%Y-%m-%d %H:%M')}._", "",
]
with open(KAGGLE_PKG_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_kaggle_readme_lines) + "\n")

with open(KAGGLE_PKG_DIR / "requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(_fallback_requirements) + "\n")

_environment_yml_lines = [
    "name: amex-risk-tier", "channels:", "  - conda-forge", "  - defaults", "dependencies:",
    "  - python=3.11", "  - pip", "  - pip:",
] + [f"      - {pkg}" for pkg in _fallback_requirements]
with open(KAGGLE_PKG_DIR / "environment.yml", "w", encoding="utf-8") as f:
    f.write("\n".join(_environment_yml_lines) + "\n")

print(f"\u2705 Saved -> {KAGGLE_PKG_DIR / 'README.md'}")
print(f"\u2705 Saved -> {KAGGLE_PKG_DIR / 'requirements.txt'}")
print(f"\u2705 Saved -> {KAGGLE_PKG_DIR / 'environment.yml'}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: LINKEDIN PROJECT SHOWCASE -- 7-SECTION DOCX + SUGGESTED CAPTION
# =============================================================================
_section("SECTION 11: LinkedIn Project Showcase -- 7-Section Docx + Suggested Caption")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_bullets(doc, items):
    for it in items:
        doc.add_paragraph(it, style="List Bullet")


li = Document()
li.add_heading("AMEX Enterprise Credit Risk Platform -- Risk Tier Classification", level=0)
li.add_paragraph("LinkedIn Project Showcase")
li.add_paragraph(
    "Every metric below is real, pulled live from this platform's own notebooks. Sections marked "
    "[EDITABLE] are personal narrative content only you can honestly write -- fill them in before "
    "posting; nothing has been invented on your behalf."
)

_add_heading(li, "1. Project Overview", level=1)
li.add_paragraph("Project Title: AMEX Enterprise Credit Risk Platform -- Risk Tier Classification")
li.add_paragraph(
    "One-line description: A 7-notebook build that translates a real, calibrated probability-of-default "
    "model into validated, deployed, and monitored risk tiers -- the second of 14 problem statements in "
    "a broader enterprise credit risk platform, built end-to-end on the real Kaggle American Express "
    "Default Prediction dataset."
)
li.add_paragraph(
    "Problem statement & objective: Turn a continuous PD score into discrete, statistically validated "
    "risk grades that drive real underwriting and pricing policy, then deploy and monitor that "
    "classification as a live service."
)

_add_heading(li, "2. Data & Tools", level=1)
_add_bullets(li, [
    "Data source: Kaggle AMEX Default Prediction competition (same real dataset as Problem 1).",
    "Key tools & technologies: scikit-learn, SciPy (statistical backtesting), FastAPI, Docker, Matplotlib.",
    "Programming language: Python 3.11.",
])

_add_heading(li, "3. Approach & Methodology", level=1)
_add_bullets(li, [
    "Step-by-step approach: policy definition -> two independently-computed tier-bucketing methods -> "
    "independent statistical validation -> live deployment with a self-test -> ongoing monitoring -> "
    "comprehensive reporting.",
    f"Reused champion model (measured, Problem 1): {_fmt_val(GLANCE['champion_model'])}.",
    "Why this approach: every displayed number is either computed live on the real dataset or an "
    "explicitly labeled, editable assumption -- including an honest, stated limitation on fair-lending "
    "testing this dataset cannot support.",
])

_add_heading(li, "4. Key Results & Impact", level=1)
_add_bullets(li, [
    f"Tiers defined / primary method: {_fmt_val(GLANCE['n_tiers'])} / {_fmt_val(GLANCE['primary_method'])}",
    f"Chi-square p-value (tier vs. actual default, real): {_fmt_val(GLANCE['chi_square_p_value'])}",
    f"Cram\u00e9r's V (effect size, real): {_fmt_val(GLANCE['cramers_v'])}",
    f"Live API self-test (real): {_fmt_val(GLANCE['api_self_test_passed'])}",
])

_add_heading(li, "5. Key Takeaways [EDITABLE -- personalize before posting]", level=1)
_add_bullets(li, [
    "[EDITABLE] What I learned: ...",
    "[EDITABLE] Challenges & how I solved them: ...",
    "[EDITABLE] Skills & growth: ...",
])

_add_heading(li, "6. Future Work", level=1)
li.add_paragraph(
    "Next in the platform's 14-problem roadmap: Phase 2 (Regulatory & Loss Provisioning) -- ECL/IFRS9-CECL, "
    "Delinquency Escalation/LGD, and Early Payment Default Detection."
)
li.add_paragraph("[EDITABLE] Additional next steps / broader applications you'd highlight: ...")

_add_heading(li, "7. Project Links [EDITABLE -- add your real links before posting]", level=1)
_add_bullets(li, [
    "[EDITABLE] GitHub Repository Link: ...",
    "[EDITABLE] Kaggle Notebook / Dataset Link: ...",
    "[EDITABLE] Dashboard / Demo Link (if any): ...",
])

linkedin_docx_path = LINKEDIN_PKG_DIR / "LinkedIn_Project_Showcase.docx"
li.save(str(linkedin_docx_path))

_caption_lines = [
    "[EDITABLE -- suggested draft, personalize before posting]", "",
    "Extended my AMEX credit risk platform with a Risk Tier Classification build -- turning a real, "
    "calibrated PD model into validated, deployed, and monitored risk tiers, with an honest, stated "
    "limitation on the fair-lending testing this dataset can't support.", "",
    f"Tiers: {_fmt_val(GLANCE['n_tiers'])} | Primary method: {_fmt_val(GLANCE['primary_method'])} | "
    f"Live API self-test: {_fmt_val(GLANCE['api_self_test_passed'])}", "",
    "#MachineLearning #CreditRisk #DataScience #FinTech #MLOps #Python #Kaggle",
]
caption_path = LINKEDIN_PKG_DIR / "linkedin_post_caption.txt"
with open(caption_path, "w", encoding="utf-8") as f:
    f.write("\n".join(_caption_lines) + "\n")

_cover_src = PILLAR_DIRS["risk_tier_reporting"] / "final_bad_rate_by_tier_chart.png"
_safe_copy("linkedin", _cover_src, LINKEDIN_PKG_DIR / "suggested_cover_image.png",
           PACKAGING_POLICY["linkedin_max_file_size_mb"])

print(f"\u2705 Saved -> {linkedin_docx_path}")
print(f"\u2705 Saved -> {caption_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: FRESH PROBLEM-2-SCOPED END-TO-END PIPELINE FLOW DIAGRAM
# =============================================================================
_section("SECTION 12: Fresh Problem-2-Scoped End-to-End Pipeline Flow Diagram")

PROBLEM_NAME = "Phase 1 \u00b7 Problem 2 -- Risk Tier Classification"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_green": "#3a9e5f", "cat_grey": "#9c9b96"}

FLOW_STAGES = [
    {"label": "Policy\n(19)", "notebooks": [19]},
    {"label": "Model Dev\n(20)", "notebooks": [20]},
    {"label": "Validation\n(21)", "notebooks": [21]},
    {"label": "Deployment\n(22)", "notebooks": [22]},
    {"label": "Monitoring\n(23)", "notebooks": [23]},
    {"label": "Reporting &\nPackaging (24-25)", "notebooks": [24, 25]},
]


def _stage_has_run(nbs):
    return all((n in NOTEBOOK_SUMMARIES) or (n == 25) for n in nbs)


fig, ax = plt.subplots(figsize=(13, 3.2), dpi=150)
ax.set_facecolor(VIZ["surface"]); fig.set_facecolor(VIZ["surface"])
ax.axis("off")
_n_stages = len(FLOW_STAGES)
_box_w, _gap = 1.8, 0.55
for i, stage in enumerate(FLOW_STAGES):
    _x = i * (_box_w + _gap)
    _color = VIZ["cat_green"] if _stage_has_run(stage["notebooks"]) else VIZ["cat_grey"]
    ax.add_patch(plt.Rectangle((_x, 0), _box_w, 1.4, facecolor=_color, edgecolor="white", linewidth=2))
    ax.text(_x + _box_w / 2, 0.7, stage["label"], ha="center", va="center", fontsize=8.5, color="white", weight="bold")
    if i < _n_stages - 1:
        ax.annotate("", xy=(_x + _box_w + _gap - 0.05, 0.7), xytext=(_x + _box_w + 0.05, 0.7),
                    arrowprops=dict(arrowstyle="->", color=VIZ["text_secondary"], lw=1.6))
ax.set_xlim(-0.2, _n_stages * (_box_w + _gap))
ax.set_ylim(-0.3, 1.9)
_legend_patches = [plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_green"], label="Complete"),
                    plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_grey"], label="Incomplete")]
ax.legend(handles=_legend_patches, loc="upper center", bbox_to_anchor=(0.5, -0.05), fontsize=8, frameon=False, ncol=2)
ax.set_title(f"{PROBLEM_NAME}\nEnd-to-End Pipeline Flow (Live Run Status)", fontsize=11, color=VIZ["text_primary"])
fig.tight_layout()
flow_diagram_path = REPO_PKG_DIR / "problem2_flow_diagram.png"
fig.savefig(flow_diagram_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)

for _pkg_dir in (GITHUB_PKG_DIR, KAGGLE_PKG_DIR):
    _dest = _pkg_dir / "docs" / "problem2_flow_diagram.png"
    _dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(flow_diagram_path, _dest)

print(f"\u2705 Saved -> {flow_diagram_path}")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: PACKAGE MANIFESTS & SIZE REPORTS
# =============================================================================
_section("SECTION 13: Package Manifests & Size Reports")

manifest_df = pd.DataFrame(PACKAGING_MANIFEST)
manifest_path = REPO_PKG_DIR / "packaging_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)


def _dir_size_mb(root: Path) -> float:
    if not root.exists():
        return 0.0
    return round(sum(f.stat().st_size for f in root.rglob("*") if f.is_file()) / 1e6, 2)


def _dir_file_count(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for f in root.rglob("*") if f.is_file())


PACKAGE_SIZE_REPORT = pd.DataFrame([
    {"package": "GitHub_Repository_Package", "files": _dir_file_count(GITHUB_PKG_DIR),
     "total_size_mb": _dir_size_mb(GITHUB_PKG_DIR), "per_file_cap_mb": GH_CAP},
    {"package": "Kaggle_Project_Package", "files": _dir_file_count(KAGGLE_PKG_DIR),
     "total_size_mb": _dir_size_mb(KAGGLE_PKG_DIR), "per_file_cap_mb": KG_CAP},
    {"package": "LinkedIn_Project_Showcase", "files": _dir_file_count(LINKEDIN_PKG_DIR),
     "total_size_mb": _dir_size_mb(LINKEDIN_PKG_DIR), "per_file_cap_mb": PACKAGING_POLICY["linkedin_max_file_size_mb"]},
])
package_size_report_path = REPO_PKG_DIR / "package_size_report.csv"
PACKAGE_SIZE_REPORT.to_csv(package_size_report_path, index=False)

_n_included = int(manifest_df["included"].sum()) if len(manifest_df) else 0
_n_excluded = int((~manifest_df["included"]).sum()) if len(manifest_df) else 0
print(manifest_df.to_string(index=False) if len(manifest_df) <= 60 else manifest_df.tail(60).to_string(index=False))
print(f"\n{PACKAGE_SIZE_REPORT.to_string(index=False)}")
print(f"\nFiles included across all packages: {_n_included}  |  Excluded (missing or over cap): {_n_excluded}")
print(f"\u2705 Saved -> {manifest_path}")
print(f"\u2705 Saved -> {package_size_report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: REPOSITORY PACKAGING READINESS CHECKLIST
# =============================================================================
_section("SECTION 14: Repository Packaging Readiness Checklist")

packaging_checklist = [
    {"dimension": "Code Quality Check Ran (Live, Every Real File)", "status": "Pass",
     "evidence": f"{_n_valid} / {_n_checked} files syntax-valid"},
    {"dimension": "GitHub Package Built & Size-Gated", "status": "Pass",
     "evidence": f"{_dir_file_count(GITHUB_PKG_DIR)} files, {_dir_size_mb(GITHUB_PKG_DIR)} MB"},
    {"dimension": "Kaggle Package Built & Size-Gated", "status": "Pass",
     "evidence": f"{_dir_file_count(KAGGLE_PKG_DIR)} files, {_dir_size_mb(KAGGLE_PKG_DIR)} MB"},
    {"dimension": "LinkedIn Showcase Built", "status": "Pass",
     "evidence": f"{_dir_file_count(LINKEDIN_PKG_DIR)} files, {_dir_size_mb(LINKEDIN_PKG_DIR)} MB"},
    {"dimension": "No Raw Competition Data Redistributed", "status": "Pass" if not PACKAGING_POLICY["copy_raw_competition_data"] else "FAIL",
     "evidence": "policy: copy_raw_competition_data=False"},
    {"dimension": "Every Packaging Decision Logged (No Silent Drops)", "status": "Pass",
     "evidence": f"{len(manifest_df)} manifest rows"},
    {"dimension": "Kaggle Notebook Mapping Documented", "status": "Pass",
     "evidence": f"{sum(1 for m in KAGGLE_NOTEBOOK_MAPPING if m['included'])} / {len(KAGGLE_NOTEBOOK_MAPPING)} mapped notebooks included"},
    {"dimension": "Problem 1's Packages Untouched (Separate Subfolder)", "status": "Pass",
     "evidence": str(REPO_PKG_DIR)},
]
packaging_checklist_df = pd.DataFrame(packaging_checklist)
packaging_checklist_path = REPO_PKG_DIR / "repository_packaging_checklist.csv"
packaging_checklist_df.to_csv(packaging_checklist_path, index=False)
print(packaging_checklist_df.to_string(index=False))
print(f"\u2705 Saved -> {packaging_checklist_path}")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: WORD REPORT -- REPOSITORY_PACKAGING_REPORT.DOCX
# =============================================================================
_section("SECTION 15: Word Report -- Repository_Packaging_Report.docx")


def _add_table_from_df(doc, df, max_rows=40):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    if len(df) > max_rows:
        doc.add_paragraph(f"... and {len(df) - max_rows} more row(s) -- see the full CSV for the complete table.")
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Repository Packaging Report -- Notebook 25 (Final, Problem 2)")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(report, "1. Package Size Report", level=1)
report.add_paragraph(
    "Every file in every package was checked against a live size cap before being copied, the same "
    "policy Notebook 18 applies for Problem 1."
)
_add_table_from_df(report, PACKAGE_SIZE_REPORT)

_add_heading(report, "2. End-to-End Pipeline Flow", level=1)
report.add_picture(str(flow_diagram_path), width=Inches(6.3))

_add_heading(report, "3. GitHub Repository Package", level=1)
report.add_paragraph(f"Built under {GITHUB_PKG_DIR}")
for _line in _repo_tree_lines[:30]:
    report.add_paragraph(_line, style="No Spacing")

_add_heading(report, "4. Kaggle Project Package -- Notebook Mapping", level=1)
_add_table_from_df(report, pd.DataFrame(KAGGLE_NOTEBOOK_MAPPING))

_add_heading(report, "5. LinkedIn Project Showcase", level=1)
report.add_paragraph(
    "A 7-section showcase document was generated at "
    f"{linkedin_docx_path.relative_to(REPO_PKG_DIR)}. Sections requiring personal narrative are clearly "
    "marked [EDITABLE]."
)

_add_heading(report, "6. Code Quality Report", level=1)
_add_table_from_df(report, code_quality_df)

_add_heading(report, "7. Packaging Manifest (What Was Included / Excluded, and Why)", level=1)
_manifest_display_df = manifest_df.copy()
for _col in ("source", "dest"):
    _manifest_display_df[_col] = _manifest_display_df[_col].apply(
        lambda p: str(Path(p).relative_to(PROJECT_ROOT)) if p and str(PROJECT_ROOT) in str(p) else p
    )
_add_table_from_df(report, _manifest_display_df, max_rows=50)

_add_heading(report, "8. Repository Packaging Readiness Checklist", level=1)
_add_table_from_df(report, packaging_checklist_df)

report_path = REPO_PKG_DIR / "Repository_Packaging_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 16: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("GitHub package has a README.md", (GITHUB_PKG_DIR / "README.md").exists())
_check("GitHub package has a LICENSE", (GITHUB_PKG_DIR / "LICENSE").exists())
_check("GitHub package has a .gitignore", (GITHUB_PKG_DIR / ".gitignore").exists())
_check("Kaggle package has a README.md", (KAGGLE_PKG_DIR / "README.md").exists())
_check("Kaggle package has an environment.yml", (KAGGLE_PKG_DIR / "environment.yml").exists())
_check("LinkedIn showcase docx exists", linkedin_docx_path.exists())
_check("No file in any package exceeds its package's size cap",
       manifest_df.loc[manifest_df["included"], "size_mb"].fillna(0).astype(float).max()
       <= max(GH_CAP, KG_CAP, PACKAGING_POLICY["linkedin_max_file_size_mb"]) if manifest_df["included"].any() else True)
_check("Raw competition CSVs were never copied into any package",
       not any(("train_data.csv" in str(m.get("dest", "")) or "test_data.csv" in str(m.get("dest", "")))
               and m.get("included") for m in PACKAGING_MANIFEST))
_check("Packaging manifest is non-empty", len(manifest_df) > 0)
_check("Code quality report covers every real Problem 2 notebook on disk",
       len(code_quality_df[code_quality_df["type"] == "notebook"]) == len(_p2_notebook_files))
_check("Problem 2 package written to its own subfolder (Problem 1 untouched)",
       REPO_PKG_DIR.name == "Problem2_Risk_Tier_Classification")

_expected_files = [code_quality_path, manifest_path, package_size_report_path, packaging_checklist_path,
                    flow_diagram_path, github_readme_path, linkedin_docx_path, caption_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 25 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 25 checks passed.")
print("\n\u2705 Section 16 complete.")


# =============================================================================
# SECTION 17: MASTER PORTFOLIO ASSEMBLY -- COMBINE PROBLEM 1 + PROBLEM 2 INTO ONE BRANDED REPO
# =============================================================================
_section("SECTION 17: Master Portfolio Assembly -- Combine Problem 1 + Problem 2 Into One Branded Repo")

# --- ASSUMPTION: this is a deliberate branding/packaging choice, not a
#     computed platform fact -- editable. Rename PLATFORM_TITLE/PLATFORM_SLUG
#     here for a different umbrella name; every future problem's own
#     packaging notebook should land its package as a new ProblemN_.../
#     subfolder alongside these, under this same PLATFORM_SLUG root. ---
PLATFORM_TITLE = "AMEX RiskIQ: Enterprise Credit Risk Platform"           # ASSUMPTION -- editable umbrella brand
PLATFORM_SLUG = "AMEX_RiskIQ_Enterprise_Credit_Risk_Platform"             # ASSUMPTION -- folder-safe version of the title

MASTER_PKG_DIR = PILLAR_DIRS["repository_packaging"] / PLATFORM_SLUG
if MASTER_PKG_DIR.exists():
    shutil.rmtree(MASTER_PKG_DIR)
MASTER_PKG_DIR.mkdir(parents=True, exist_ok=True)

MASTER_GITHUB_DIR = MASTER_PKG_DIR / "GitHub_Repository_Package"
MASTER_KAGGLE_DIR = MASTER_PKG_DIR / "Kaggle_Project_Package"
MASTER_LINKEDIN_DIR = MASTER_PKG_DIR / "LinkedIn_Project_Showcase"
for _d in (MASTER_GITHUB_DIR, MASTER_KAGGLE_DIR, MASTER_LINKEDIN_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Problem 1's real packages, built by Notebook 18, live directly under the
#     shared repository_packaging pillar this notebook also uses. If Notebook
#     18 has not been run yet, that is recorded honestly below -- never faked. ---
PROBLEM1_GITHUB_SRC = PILLAR_DIRS["repository_packaging"] / "GitHub_Repository_Package"
PROBLEM1_KAGGLE_SRC = PILLAR_DIRS["repository_packaging"] / "Kaggle_Project_Package"
PROBLEM1_LINKEDIN_SRC = PILLAR_DIRS["repository_packaging"] / "LinkedIn_Project_Showcase"

MASTER_MANIFEST = []


def _safe_copytree(package: str, src_dir: Path, dest_dir: Path, label: str) -> bool:
    """Copy an already-built, already-verified package directory wholesale --
    the same honest-manifest discipline as _safe_copy, applied to a whole
    real folder instead of a single file. Never fabricates content: a
    missing source gets an honest placeholder README, not invented files."""
    if src_dir.exists() and src_dir.is_dir() and any(src_dir.iterdir()):
        shutil.copytree(src_dir, dest_dir, dirs_exist_ok=True)
        _size_mb = round(sum(f.stat().st_size for f in dest_dir.rglob("*") if f.is_file()) / 1e6, 2)
        MASTER_MANIFEST.append({"package": package, "source": str(src_dir), "dest": str(dest_dir),
                                 "included": True, "size_mb": _size_mb,
                                 "reason": f"real, already-built {label} package (directory copy)"})
        print(f"\u2705 {label} -> {dest_dir}")
        return True
    dest_dir.mkdir(parents=True, exist_ok=True)
    with open(dest_dir / "README.md", "w", encoding="utf-8") as f:
        f.write(f"# {label}\n\nNot yet built this run -- run the notebook that builds this package first, "
                f"then re-run this notebook to include it here.\n")
    MASTER_MANIFEST.append({"package": package, "source": str(src_dir), "dest": str(dest_dir),
                             "included": False, "size_mb": None,
                             "reason": f"{label} package not found -- source notebook has not been run yet"})
    print(f"\u26a0\ufe0f  {label} not found at {src_dir} -- placeholder written, nothing fabricated")
    return False


PROBLEM1_GITHUB_INCLUDED = _safe_copytree("github", PROBLEM1_GITHUB_SRC, MASTER_GITHUB_DIR / "Problem1_Default_Prediction", "Problem 1 (Default Prediction) -- GitHub package")
PROBLEM2_GITHUB_INCLUDED = _safe_copytree("github", GITHUB_PKG_DIR, MASTER_GITHUB_DIR / "Problem2_Risk_Tier_Classification", "Problem 2 (Risk Tier Classification) -- GitHub package")

PROBLEM1_KAGGLE_INCLUDED = _safe_copytree("kaggle", PROBLEM1_KAGGLE_SRC, MASTER_KAGGLE_DIR / "Problem1_Default_Prediction", "Problem 1 (Default Prediction) -- Kaggle package")
PROBLEM2_KAGGLE_INCLUDED = _safe_copytree("kaggle", KAGGLE_PKG_DIR, MASTER_KAGGLE_DIR / "Problem2_Risk_Tier_Classification", "Problem 2 (Risk Tier Classification) -- Kaggle package")

PROBLEM1_LINKEDIN_INCLUDED = _safe_copytree("linkedin", PROBLEM1_LINKEDIN_SRC, MASTER_LINKEDIN_DIR / "Problem1_Default_Prediction", "Problem 1 (Default Prediction) -- LinkedIn showcase")
PROBLEM2_LINKEDIN_INCLUDED = _safe_copytree("linkedin", LINKEDIN_PKG_DIR, MASTER_LINKEDIN_DIR / "Problem2_Risk_Tier_Classification", "Problem 2 (Risk Tier Classification) -- LinkedIn showcase")

print(f"\nMaster combined package root: {MASTER_PKG_DIR}")
print("\n\u2705 Section 17 complete.")


# =============================================================================
# SECTION 18: MASTER GITHUB REPO -- REAL PLATFORM-WIDE HEADLINE METRICS & ROOT README.md
# =============================================================================
_section("SECTION 18: Master GitHub Repo -- Real Platform-Wide Headline Metrics & Root README.md")

PROBLEM1_GLANCE = {
    "champion_model": _g(5, "champion_model"),
    "champion_holdout_auc": _g(5, "champion_metrics", "holdout_auc"),
    "champion_holdout_amex_metric": _g(5, "champion_metrics", "holdout_amex_metric"),
    "total_rwa_usd": _g(8, "total_rwa_usd"),
    "total_ecl_usd": _g(8, "total_ecl_usd"),
    "api_self_test_passed": _g(10, "api_self_test_passed"),
    "monitoring_alerts": _g(12, "n_alerts"),
    "base_5yr_roi_pct": _g(14, "base_5yr_roi_pct"),
    "core_pipeline_completion_pct": _g(17, "core_pipeline_completion_pct"),
}
for _k, _v in PROBLEM1_GLANCE.items():
    print(f"  problem1.{_k:32s}: {_v}")

_PROBLEM_ROWS = [
    {"problem": "Problem 1 -- Credit Default Prediction", "notebooks": "01-18 (18 notebooks)",
     "status": "Included in this build" if PROBLEM1_GITHUB_INCLUDED else "Not yet built -- run Notebook 18",
     "headline_metric": f"Champion {_fmt_val(PROBLEM1_GLANCE['champion_model'])}, holdout AUC "
                         f"{_fmt_val(PROBLEM1_GLANCE['champion_holdout_auc'])}",
     "folder": "Problem1_Default_Prediction/"},
    {"problem": "Problem 2 -- Risk Tier Classification", "notebooks": "19-25 (7 notebooks)",
     "status": "Included in this build",
     "headline_metric": f"{_fmt_val(GLANCE['n_tiers'])} tiers, chi-square p-value "
                         f"{_fmt_val(GLANCE['chi_square_p_value'])}",
     "folder": "Problem2_Risk_Tier_Classification/"},
]

_master_repo_tree_lines = [f"{MASTER_GITHUB_DIR.name}/"] + _render_tree(MASTER_GITHUB_DIR, max_entries=10)

_master_readme_lines = [
    f"# {PLATFORM_TITLE}",
    "",
    "[![Python 3.11](https://img.shields.io/badge/python-3.11-blue.svg)]() "
    "[![CRISP-DM](https://img.shields.io/badge/methodology-CRISP--DM-informational.svg)]() "
    "[![License: MIT](https://img.shields.io/badge/license-MIT-green.svg)](LICENSE)",
    "",
    "An enterprise-grade, end-to-end credit risk platform built entirely on the real Kaggle **American "
    "Express Default Prediction** dataset -- multiple real problem statements, each its own fully "
    "validated, deployed, and monitored build, combined here as one portfolio repository. Every result "
    "in every subfolder is either computed live by that problem's own notebooks or an explicitly "
    "labeled, editable assumption -- nothing in this repository is fabricated or guessed.",
    "",
    "## Problems In This Repository",
    "",
    "| Problem | Notebooks | Status | Headline Result (Real) | Folder |",
    "|---|---|---|---|---|",
] + [
    f"| {r['problem']} | {r['notebooks']} | {r['status']} | {r['headline_metric']} | `{r['folder']}` |"
    for r in _PROBLEM_ROWS
] + [
    "",
    "Each problem's folder is a **complete, self-contained package** -- its own notebooks/, src/, "
    "reports/, docs/, assets/, models/, data/, README.md, LICENSE, and requirements.txt. Open either "
    "folder's own README.md for that problem's full write-up, methodology, and real measured results.",
    "",
    "## Platform Roadmap",
    "",
    "This is Phase 1 of a planned 14-problem enterprise credit risk platform. Problems 1 and 2 are real "
    "and complete as of this build; later phases (regulatory & loss provisioning, delinquency escalation, "
    "early payment default detection, and beyond) are planned future additions to this same repository, "
    "each landing as its own `ProblemN_.../` subfolder alongside the ones already here.",
    "",
    "## Repository Structure",
    "",
    "This tree is real -- generated by walking this exact repository root, not hand-typed:",
    "",
    "```",
] + _master_repo_tree_lines + [
    "```",
    "",
    "## How To Run",
    "",
    "1. Download the real Kaggle American Express Default Prediction competition data "
    "(https://www.kaggle.com/competitions/amex-default-prediction).",
    "2. Start with `Problem1_Default_Prediction/notebooks/01_business_understanding.ipynb` and run "
    "Problem 1's 18 notebooks in numeric order.",
    "3. Then run `Problem2_Risk_Tier_Classification/notebooks/19_risk_tier_business_understanding.ipynb` "
    "through `25_risk_tier_packaging.ipynb` -- Problem 2 depends on Problem 1's real trained champion model.",
    "",
    "## License",
    "",
    "MIT -- see `LICENSE`.",
    "",
    f"_Generated by 25_risk_tier_packaging.ipynb, {datetime.now().strftime('%Y-%m-%d %H:%M')}._",
    "",
]
MASTER_README_CONTENT = "\n".join(_master_readme_lines)
master_readme_path = MASTER_GITHUB_DIR / "README.md"
with open(master_readme_path, "w", encoding="utf-8") as f:
    f.write(MASTER_README_CONTENT)

with open(MASTER_GITHUB_DIR / "LICENSE", "w", encoding="utf-8") as f:
    f.write(_license_text)

with open(MASTER_GITHUB_DIR / ".gitignore", "w", encoding="utf-8") as f:
    f.write("\n".join(_gitignore_lines) + "\n")

# --- Root requirements.txt: a real union of Problem 1's actual copied list
#     (if present) and Problem 2's documented baseline -- never invented. ---
_problem1_requirements_path = MASTER_GITHUB_DIR / "Problem1_Default_Prediction" / "requirements.txt"
_problem1_requirements = []
if _problem1_requirements_path.exists():
    with open(_problem1_requirements_path, "r", encoding="utf-8") as f:
        _problem1_requirements = [l.strip() for l in f if l.strip()]
_combined_requirements = sorted(set(_problem1_requirements) | set(_fallback_requirements), key=str.lower)
with open(MASTER_GITHUB_DIR / "requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(_combined_requirements) + "\n")

print(f"\u2705 Saved -> {master_readme_path}  ({len(MASTER_README_CONTENT):,} characters)")
print("\n\u2705 Section 18 complete.")


# =============================================================================
# SECTION 19: MASTER KAGGLE PROJECT -- ROOT README.md, requirements.txt & environment.yml
# =============================================================================
_section("SECTION 19: Master Kaggle Project -- Root README.md, requirements.txt & environment.yml")

_master_kaggle_readme_lines = [
    f"# {PLATFORM_TITLE} (Kaggle Project Package)",
    "",
    "## Overview",
    "",
    "A multi-problem enterprise credit risk platform built on the real Kaggle American Express Default "
    "Prediction competition dataset. Each problem below is a complete, independently runnable package.",
    "", "## Problems In This Project", "",
    "| Problem | Status | Headline Result (Real) | Folder |",
    "|---|---|---|---|",
] + [f"| {r['problem']} | {r['status']} | {r['headline_metric']} | `{r['folder']}` |" for r in _PROBLEM_ROWS] + [
    "", "## Dataset", "",
    "Kaggle competition: American Express Default Prediction "
    "(https://www.kaggle.com/competitions/amex-default-prediction). Raw data is not redistributed here.",
    "", "## How to Run", "",
    "1. Install dependencies from `requirements.txt` (or `environment.yml` for conda).",
    "2. Open each problem's own `notebooks/NOTEBOOK_MAPPING.md` for the exact run order.",
    "", "## Acknowledgements", "",
    "Built on the Kaggle American Express Default Prediction competition dataset.",
    "", f"_Generated by 25_risk_tier_packaging.ipynb, {datetime.now().strftime('%Y-%m-%d %H:%M')}._", "",
]
with open(MASTER_KAGGLE_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write("\n".join(_master_kaggle_readme_lines) + "\n")

with open(MASTER_KAGGLE_DIR / "requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(_combined_requirements) + "\n")

_master_environment_yml_lines = [
    "name: amex-riskiq-platform", "channels:", "  - conda-forge", "  - defaults", "dependencies:",
    "  - python=3.11", "  - pip", "  - pip:",
] + [f"      - {pkg}" for pkg in _combined_requirements]
with open(MASTER_KAGGLE_DIR / "environment.yml", "w", encoding="utf-8") as f:
    f.write("\n".join(_master_environment_yml_lines) + "\n")

print(f"\u2705 Saved -> {MASTER_KAGGLE_DIR / 'README.md'}")
print(f"\u2705 Saved -> {MASTER_KAGGLE_DIR / 'requirements.txt'}")
print(f"\u2705 Saved -> {MASTER_KAGGLE_DIR / 'environment.yml'}")
print("\n\u2705 Section 19 complete.")


# =============================================================================
# SECTION 20: MASTER LINKEDIN SHOWCASE -- UNIFIED PLATFORM DOCX + SUGGESTED CAPTION
# =============================================================================
_section("SECTION 20: Master LinkedIn Showcase -- Unified Platform Docx + Suggested Caption")

plat = Document()
plat.add_heading(PLATFORM_TITLE, level=0)
plat.add_paragraph("LinkedIn Project Showcase -- Combined Portfolio (All Problems Built So Far)")
plat.add_paragraph(
    "Every metric below is real, pulled live from this platform's own notebooks. Sections marked "
    "[EDITABLE] are personal narrative content only you can honestly write -- fill them in before "
    "posting; nothing has been invented on your behalf."
)

_add_heading(plat, "1. Project Overview", level=1)
plat.add_paragraph(f"Project Title: {PLATFORM_TITLE}")
plat.add_paragraph(
    "One-line description: An enterprise-grade, end-to-end credit risk platform built on the real Kaggle "
    "American Express Default Prediction dataset -- each problem statement is its own fully validated, "
    "deployed, and monitored build, combined here as one portfolio."
)
plat.add_paragraph(
    "Problem statement & objective: Go from raw transaction-level credit data to a production-grade "
    "default prediction model, then extend that model into real, validated business decisions -- starting "
    "with risk tier classification for underwriting and pricing."
)

_add_heading(plat, "2. Data & Tools", level=1)
_add_bullets(plat, [
    "Data source: Kaggle American Express Default Prediction competition (real, not synthetic).",
    "Key tools & technologies: Polars, scikit-learn, XGBoost/LightGBM/CatBoost, SHAP, FastAPI, Docker, "
    "Power BI, Matplotlib.",
    "Programming language: Python 3.11.",
])

_add_heading(plat, "3. Approach & Methodology", level=1)
_add_bullets(plat, [
    "Problem 1 (18 notebooks): CRISP-DM pipeline from business understanding through data engineering, "
    "modeling, explainability, model risk management, Basel/IFRS9 mapping, MLOps, deployment, monitoring, "
    "dashboards, and executive reporting.",
    "Problem 2 (7 notebooks): extends Problem 1's real, calibrated PD score into validated, deployed, and "
    "monitored risk tiers for underwriting and pricing policy.",
    "Why this approach: every displayed number across both problems is either computed live on the real "
    "dataset or an explicitly labeled, editable assumption -- nothing in this platform is fabricated.",
])

_add_heading(plat, "4. Key Results & Impact (Real, Measured)", level=1)
_add_bullets(plat, [
    f"Problem 1 -- Champion model: {_fmt_val(PROBLEM1_GLANCE['champion_model'])}, holdout AUC: "
    f"{_fmt_val(PROBLEM1_GLANCE['champion_holdout_auc'])}",
    f"Problem 1 -- Live API self-test: {_fmt_val(PROBLEM1_GLANCE['api_self_test_passed'])}",
    f"Problem 2 -- Tiers defined / primary method: {_fmt_val(GLANCE['n_tiers'])} / {_fmt_val(GLANCE['primary_method'])}",
    f"Problem 2 -- Chi-square p-value (tier vs. actual default): {_fmt_val(GLANCE['chi_square_p_value'])}",
    f"Problem 2 -- Live API self-test: {_fmt_val(GLANCE['api_self_test_passed'])}",
])

_add_heading(plat, "5. Key Takeaways [EDITABLE -- personalize before posting]", level=1)
_add_bullets(plat, [
    "[EDITABLE] What I learned: ...",
    "[EDITABLE] Challenges & how I solved them: ...",
    "[EDITABLE] Skills & growth: ...",
])

_add_heading(plat, "6. Future Work", level=1)
plat.add_paragraph(
    "Next in the platform's 14-problem roadmap: Phase 2 (Regulatory & Loss Provisioning) -- ECL/IFRS9-CECL, "
    "Delinquency Escalation/LGD, and Early Payment Default Detection, each landing as its own subfolder in "
    "this same repository."
)
plat.add_paragraph("[EDITABLE] Additional next steps / broader applications you'd highlight: ...")

_add_heading(plat, "7. Project Links [EDITABLE -- add your real links before posting]", level=1)
_add_bullets(plat, [
    "[EDITABLE] GitHub Repository Link: ...",
    "[EDITABLE] Kaggle Notebook / Dataset Link: ...",
    "[EDITABLE] Dashboard / Demo Link (if any): ...",
])

platform_linkedin_docx_path = MASTER_LINKEDIN_DIR / "Platform_Project_Showcase.docx"
plat.save(str(platform_linkedin_docx_path))

_platform_caption_lines = [
    "[EDITABLE -- suggested draft, personalize before posting]", "",
    f"Built {PLATFORM_TITLE} end-to-end on the real Kaggle American Express Default Prediction dataset -- "
    "starting with a full credit default prediction pipeline, then extending it into validated, deployed, "
    "and monitored risk tier classification for real underwriting and pricing decisions.", "",
    f"Problem 1 champion AUC: {_fmt_val(PROBLEM1_GLANCE['champion_holdout_auc'])} | "
    f"Problem 2 tiers: {_fmt_val(GLANCE['n_tiers'])} | Both live API self-tests: "
    f"{_fmt_val(PROBLEM1_GLANCE['api_self_test_passed'])} / {_fmt_val(GLANCE['api_self_test_passed'])}", "",
    "#MachineLearning #CreditRisk #DataScience #FinTech #MLOps #Python #Kaggle",
]
platform_caption_path = MASTER_LINKEDIN_DIR / "platform_post_caption.txt"
with open(platform_caption_path, "w", encoding="utf-8") as f:
    f.write("\n".join(_platform_caption_lines) + "\n")

_platform_cover_src = PILLAR_DIRS["risk_tier_reporting"] / "final_bad_rate_by_tier_chart.png"
if _platform_cover_src.exists():
    shutil.copy2(_platform_cover_src, MASTER_LINKEDIN_DIR / "suggested_cover_image.png")

print(f"\u2705 Saved -> {platform_linkedin_docx_path}")
print(f"\u2705 Saved -> {platform_caption_path}")
print("\n\u2705 Section 20 complete.")


# =============================================================================
# SECTION 21: MASTER PACKAGE MANIFEST, READINESS CHECKLIST & VERIFICATION
# =============================================================================
_section("SECTION 21: Master Package Manifest, Readiness Checklist & Verification")

master_manifest_df = pd.DataFrame(MASTER_MANIFEST)
master_manifest_path = MASTER_PKG_DIR / "master_packaging_manifest.csv"
master_manifest_df.to_csv(master_manifest_path, index=False)
print(master_manifest_df.to_string(index=False))
print(f"\u2705 Saved -> {master_manifest_path}")

master_checklist = [
    {"dimension": "Master Repository Root Built", "status": "Pass", "evidence": str(MASTER_PKG_DIR.name)},
    {"dimension": "Problem 1 Subfolder Present", "status": "Pass" if PROBLEM1_GITHUB_INCLUDED else "Not Yet Built",
     "evidence": ("Run 18_repository_packaging.ipynb, then re-run this notebook" if not PROBLEM1_GITHUB_INCLUDED
                  else f"{_dir_file_count(MASTER_GITHUB_DIR / 'Problem1_Default_Prediction')} files")},
    {"dimension": "Problem 2 Subfolder Present", "status": "Pass",
     "evidence": f"{_dir_file_count(MASTER_GITHUB_DIR / 'Problem2_Risk_Tier_Classification')} files"},
    {"dimension": "Root README.md, LICENSE, .gitignore Present", "status": "Pass",
     "evidence": "all three exist"},
    {"dimension": "Master Kaggle Project Root Built", "status": "Pass", "evidence": str(MASTER_KAGGLE_DIR.name)},
    {"dimension": "Master LinkedIn Platform Showcase Built", "status": "Pass",
     "evidence": platform_linkedin_docx_path.name},
    {"dimension": "Every Master Assembly Decision Logged (No Silent Drops)", "status": "Pass",
     "evidence": f"{len(master_manifest_df)} manifest rows"},
]
master_checklist_df = pd.DataFrame(master_checklist)
master_checklist_path = MASTER_PKG_DIR / "master_packaging_checklist.csv"
master_checklist_df.to_csv(master_checklist_path, index=False)
print(f"\n{master_checklist_df.to_string(index=False)}")
print(f"\u2705 Saved -> {master_checklist_path}")

_master_checks_passed = True


def _master_check(label, condition, detail=""):
    global _master_checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _master_checks_passed = False
        print(f"\u274c {label}  {detail}")


_master_check("Master package root exists", MASTER_PKG_DIR.exists())
_master_check("Master GitHub root README.md exists", master_readme_path.exists() and master_readme_path.stat().st_size > 0)
_master_check("Master GitHub root LICENSE exists", (MASTER_GITHUB_DIR / "LICENSE").exists())
_master_check("Master GitHub root .gitignore exists", (MASTER_GITHUB_DIR / ".gitignore").exists())
_master_check("Problem 2 GitHub subfolder is present and non-empty",
              _dir_file_count(MASTER_GITHUB_DIR / "Problem2_Risk_Tier_Classification") > 0)
_master_check("Master Kaggle root README.md exists", (MASTER_KAGGLE_DIR / "README.md").exists())
_master_check("Master LinkedIn showcase docx exists", platform_linkedin_docx_path.exists())
_master_check("Master manifest is non-empty", len(master_manifest_df) > 0)

if not _master_checks_passed:
    raise RuntimeError("One or more master portfolio assembly checks failed. See \u274c lines above.")

print("\nAll master portfolio assembly checks passed.")
print("\n\u2705 Section 21 complete.")


# =============================================================================
# SECTION 22: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 22: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "pyflakes_available": _HAS_PYFLAKES,
}
performance_report_path = ARTIFACTS_DIR / "notebook_25_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 22 complete.")


# =============================================================================
# SECTION 23: WRITE NOTEBOOK 25 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 23: Write Notebook 25 Summary Artifact")

notebook_25_summary = {
    "notebook": "25_risk_tier_packaging", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 2, "problem_name": "Risk Tier Classification",
    "github_package_files": _dir_file_count(GITHUB_PKG_DIR), "github_package_size_mb": _dir_size_mb(GITHUB_PKG_DIR),
    "kaggle_package_files": _dir_file_count(KAGGLE_PKG_DIR), "kaggle_package_size_mb": _dir_size_mb(KAGGLE_PKG_DIR),
    "linkedin_package_files": _dir_file_count(LINKEDIN_PKG_DIR),
    "code_quality_files_checked": _n_checked, "code_quality_files_valid": _n_valid,
    "packaging_manifest_included": _n_included, "packaging_manifest_excluded": _n_excluded,
    "platform_title": PLATFORM_TITLE, "platform_slug": PLATFORM_SLUG,
    "master_package_dir": str(MASTER_PKG_DIR),
    "master_github_files": _dir_file_count(MASTER_GITHUB_DIR), "master_github_size_mb": _dir_size_mb(MASTER_GITHUB_DIR),
    "master_kaggle_files": _dir_file_count(MASTER_KAGGLE_DIR), "master_kaggle_size_mb": _dir_size_mb(MASTER_KAGGLE_DIR),
    "master_linkedin_files": _dir_file_count(MASTER_LINKEDIN_DIR),
    "problem1_included_in_master": PROBLEM1_GITHUB_INCLUDED,
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
    "master_output_files": {
        "README.md": str(master_readme_path), "master_packaging_manifest.csv": str(master_manifest_path),
        "master_packaging_checklist.csv": str(master_checklist_path),
        "Platform_Project_Showcase.docx": str(platform_linkedin_docx_path),
    },
}
nb25_summary_path = ARTIFACTS_DIR / "notebook_25_summary.json"
with open(nb25_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_25_summary, f, indent=2)
print(f"\u2705 Saved -> {nb25_summary_path}")
print("\n\u2705 Section 23 complete.")


# =============================================================================
# SECTION 24: FINAL COMPLETION SUMMARY -- PROBLEM 2 (NOTEBOOKS 19-25) & MASTER PORTFOLIO
# =============================================================================
_section("SECTION 24: Notebook 25 Complete -- Problem 2 Build Finished & Master Portfolio Assembled")

print("NOTEBOOK 25: RISK TIER REPOSITORY PACKAGING -- COMPLETE")
print(f"  Problem 2's own GitHub package   : {_dir_file_count(GITHUB_PKG_DIR)} files, {_dir_size_mb(GITHUB_PKG_DIR)} MB -> {GITHUB_PKG_DIR}")
print(f"  Problem 2's own Kaggle package    : {_dir_file_count(KAGGLE_PKG_DIR)} files, {_dir_size_mb(KAGGLE_PKG_DIR)} MB -> {KAGGLE_PKG_DIR}")
print(f"  Problem 2's own LinkedIn showcase : {_dir_file_count(LINKEDIN_PKG_DIR)} files -> {LINKEDIN_PKG_DIR}")
print(f"  Code quality                     : {_n_valid} / {_n_checked} files syntax-valid")
print(f"\n  MASTER PORTFOLIO -- \"{PLATFORM_TITLE}\"")
print(f"  Master repo root      : {MASTER_PKG_DIR}")
print(f"  -> GitHub package     : {_dir_file_count(MASTER_GITHUB_DIR)} files, {_dir_size_mb(MASTER_GITHUB_DIR)} MB "
      f"(Problem1_Default_Prediction/ {'included' if PROBLEM1_GITHUB_INCLUDED else 'NOT YET BUILT -- run Notebook 18'}, "
      f"Problem2_Risk_Tier_Classification/ included)")
print(f"  -> Kaggle package     : {_dir_file_count(MASTER_KAGGLE_DIR)} files, {_dir_size_mb(MASTER_KAGGLE_DIR)} MB")
print(f"  -> LinkedIn showcase  : {_dir_file_count(MASTER_LINKEDIN_DIR)} files -> {platform_linkedin_docx_path.name}")
print(f"  Files produced      : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb25_summary_path]:
    print(f"    - {_p.name}")
print(f"\n  This is Notebook 25 of 25 -- Phase 1, Problem 2 (Risk Tier Classification) is complete.")
if PROBLEM1_GITHUB_INCLUDED:
    print(f"  Both Problem 1 and Problem 2 are now packaged together as ONE repository, ONE Kaggle project, "
          f"and ONE LinkedIn showcase under the \"{PLATFORM_TITLE}\" brand -- ready to push/share as a single portfolio.")
else:
    print(f"  Problem 2 is packaged under \"{PLATFORM_TITLE}\" now; run 18_repository_packaging.ipynb and "
          f"then re-run this notebook to fold Problem 1 in alongside it.")
print(f"  Re-run Notebook 24 (Comprehensive Reporting) any time to see Problem 2's live overall status.")
print("\n\u2705 Ready to share.")
